In [1]:
import torch
import json
import io
import numpy as np
import scipy.special as sp

import pickle as pkl
import zlib
import base64

/Users/aleksei/.local/share/virtualenvs/kutulu-_n6nfavE/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from hydra import initialize, compose
from omegaconf import OmegaConf

In [3]:
import sys
sys.path.insert(0, '..')

In [4]:
# import sys
# sys.path.insert(0, '/Users/aleksei/projects/code-of-kutulu-client')

In [5]:
from tests.utils import calculate_entities

In [6]:
from src.envs.agents.ppo_agent import PPOAgent

from src.envs.kutulu_observer import KutuluClosestObserver, KutuluClosestExtObserver
from src.envs.kutulu_world import KutuluWorldEnv
from src.game.template import (
    CELL_WALL, DEFAULT_KUTULU_ACTIONS, EXTENDED_KUTULU_ACTIONS,
    PPOConvSolver, PPOConvExtSolver, DQNConvSolver, DQNSolver,
    PPOConvExtDeepSolver,
)
from src.game.template import MOVE_REL_POS, REL_POSITIONS
from src.envs.agent_validator import AgentValidator
from src.game.template import DEFAULT_KUTULU_ACTIONS, EXTENDED_KUTULU_ACTIONS, UnreachedPositionError


In [7]:
# experiment = '0679b48e9b374e0f9c82535d01448c51'

In [8]:
# experiment = '05abe073de06428e896fcd880c9f3eac'

In [9]:
# experiment = '6d75a83e942740d2b077f6e815942a15'

In [10]:
# experiment = 'cd5d12509c5243f8b5fd8eb2fef0fdd9'

In [11]:
# 148 / 314
experiment = '9ad30a911ecb4619b530c09a000f80de'

In [12]:
# 5 / 314
experiment = 'c1ef7ae2d0e84408bee832b580b2b5d8'

In [13]:
# 23 / 314
experiment = 'd00fa77b3b3d4a4d9a0b1b7203b7de47'

In [14]:
# 4 / 314
experiment = '08d2f5ac186a44fb892f43181e199273'

In [15]:
mode = 'ppo_conv'

In [16]:
# 3 / 314
experiment = 'b522544f675143a6965c89d20500e446'
mode = 'ppo_conv_ext'

In [17]:
# 
experiment = '97ec2cbfadc643ccac11b233688554b0'
mode = 'ppo_conv_ext'

In [18]:
# 1 / 314
experiment = '078954480ae541b5b056123f5e6661e7'
mode = 'ppo_conv_ext'

In [19]:
# ? / 314
experiment = '3aa3b833fc4f461aafb599c531914eef'
mode = 'ppo_conv_ext'

In [20]:
# ? / 314
experiment = 'fcdd7a3dab614c44ba055de4d1e252e8'
mode = 'ppo_conv_ext'

In [21]:
# 1 / 314
experiment = 'cc4daeb610014bec93a1ec7b47c946d3'
mode = 'ppo_conv_ext'

In [22]:
# 
experiment = '843af99938034df0b714f52c4ee3243f'
mode = 'ppo_conv_ext'

In [23]:
# 
experiment = '39d43124601841dd9ec09ffdc0515a0d'
mode = 'ppo_conv_ext'

In [24]:
# 
experiment = 'cf4f05b2346149bab4c94ca6bef899f0'
mode = 'ppo_conv_ext'

In [25]:
# 
experiment = '3f174e4fe6f64fe8af02c1b24f3f2bb8'
mode = 'ppo_conv_ext'

In [26]:
# ? / 314
experiment = '7f0fd165b0874c3a9f7cb391daa5f4d1'
mode = 'ppo_conv_ext'

In [27]:
# 2 / 314
experiment = '5ed54fc318654e68b14fe1422693f950'
mode = 'ppo_conv_ext'

In [28]:
# 1 / 314
experiment = '9f0f7c6074f84ff8b5778b3d82ebee01'
mode = 'ppo_conv_ext'

In [29]:
# 1 / 314
experiment = 'b1c35809c181481d90fab3e07e54bcf4'
mode = 'ppo_conv_ext_deep'

In [30]:
config_path = f'../../kutulu_artifacts/mlflow_artifacts/{experiment}/artifacts/hydra_config'
checkpoint_dir = f'../../kutulu_artifacts/mlflow_artifacts/{experiment}/artifacts/models/agent_0/final'

In [31]:
with initialize(version_base=None, config_path=config_path):
    cfg = compose(config_name="config")

In [32]:
info = OmegaConf.to_container(cfg.agent, resolve=True)
del info['type']

In [33]:
agent = PPOAgent(**info)

In [34]:
agent.model.load_state_dict(torch.load(f"{checkpoint_dir}/model.pt"))

<All keys matched successfully>

In [35]:
agent.train = True

In [36]:
av = AgentValidator(EXTENDED_KUTULU_ACTIONS)
av_plan = AgentValidator(EXTENDED_KUTULU_ACTIONS, player_params=(100, 1, 0))

In [37]:
# av.check_entity_nearby(agent, 'SLASHER', n_min=2, n_max=2, verbose=True)

In [38]:
av.check_entity_nearby(agent, 'SLASHER', n_min=3, n_max=3)

(1.0, 0.0, 1, 0.0, 0.0)

In [39]:
av.check_entity_nearby(agent, 'EXPLORER', n_min=2, n_max=3)

(1.0, 0.0, 4, 0.0, 0.0)

In [40]:
av_plan.check_entity_nearby(agent, 'EXPLORER', n_min=1, n_max=2)

(1.0, 0.0, 4, 0.0, 0.0)

In [41]:
av_plan.check_entity_nearby(agent, 'EXPLORER', n_min=3, n_max=3)

(1.0, 0.0, 4, 0.0, 0.0)

In [42]:
av.check_entity_nearby(agent, 'WANDERER', n_min=1, n_max=2)

(1.0, 0.0, 1, 0.0, 0.0)

In [43]:
av.check_entity_nearby(agent, 'WANDERER', n_min=1, n_max=2, env_types=['corner'])

(1.0, 0.0, 4, 0.0, 0.0)

In [44]:
av.check_entity_nearby(agent, 'WANDERER', n_min=1, n_max=2, env_types=['coridor'])

(1.0, 0.0, 4, 0.0, 0.0)

In [45]:
agent.train = False

In [46]:
weights = {}
for k,v in agent.model.state_dict().items():
    weights[k] = v.detach().numpy()
    print(k, v.shape)

encoder.mconv_list.0.ones_kernel torch.Size([1, 1, 3, 3])
encoder.mconv_list.0.conv.weight torch.Size([8, 21, 3, 3])
encoder.mconv_list.0.conv.bias torch.Size([8])
encoder.mconv_list.1.ones_kernel torch.Size([1, 1, 5, 5])
encoder.mconv_list.1.conv.weight torch.Size([8, 21, 5, 5])
encoder.mconv_list.1.conv.bias torch.Size([8])
encoder.ln_list.0.weight torch.Size([8])
encoder.ln_list.0.bias torch.Size([8])
encoder.ln_list.1.weight torch.Size([8])
encoder.ln_list.1.bias torch.Size([8])
encoder.fc_list.0.weight torch.Size([16, 8])
encoder.fc_list.0.bias torch.Size([16])
encoder.fc_list.1.weight torch.Size([16, 8])
encoder.fc_list.1.bias torch.Size([16])
actor.weight torch.Size([8, 16])
actor.bias torch.Size([8])
critic.weight torch.Size([1, 16])
critic.bias torch.Size([1])
terminator.weight torch.Size([1, 16])
terminator.bias torch.Size([1])
occupation_head.weight torch.Size([1, 16])
occupation_head.bias torch.Size([1])


In [47]:
env = KutuluWorldEnv('', '', 1, actions=EXTENDED_KUTULU_ACTIONS)
env.map = [
    '###########',
    '#.........#',
    '#.#.#.#.#.#',
    '#.........#',
    '#.#.#.#.#.#',
    '#.........#',
    '###########',
]
env.width = len(env.map[0])
env.height = len(env.map)

In [48]:
player_pos = (3, 3)
explorers = [(5, 3), (1, 3)]
wanderers = [(3, 1, 1), (3, 5, 1), (3, 2, 0)]

entities = calculate_entities(player_pos, explorers, wanderers)
agent.set_env(env)
env._set_entities(entities)
env._set_players(entities, set_ids=True)

state = agent.observer.get_state(0)
# test_data = agent.episode_buffer.encode_states([state], return_tensors=False)
tensor_data = agent.episode_buffer.state_encoder.encode_states([state], return_tensors=True)

In [49]:
info = {
    'width': env.width,
    'height': env.height,
    'lines': env.map,
}

# solver = PPOConvExtSolver(info, EXTENDED_KUTULU_ACTIONS, weights, size=agent.size)

In [50]:
USED_ACTIONS = EXTENDED_KUTULU_ACTIONS
checkpoint_data = weights
SIZE = agent.size
explicit_action_mask = agent.explicit_action_mask

In [51]:
if mode == 'qlearning':
    solver = QlearningSolver(info, USED_ACTIONS, checkpoint_data)
elif mode == 'dqn_ext':
    solver = DQNSolver(info, USED_ACTIONS, checkpoint_data)
elif mode == 'dqn_by_kind':
    solver = DQNByKindSolver(info, USED_ACTIONS, checkpoint_data)
elif mode == 'dqn_conv':
    solver = DQNConvSolver(info, USED_ACTIONS, checkpoint_data, SIZE)
elif mode == 'ppo_conv':
    solver = PPOConvSolver(info, USED_ACTIONS, checkpoint_data, SIZE)
elif mode == 'ppo_conv_ext':
    solver = PPOConvExtSolver(info, USED_ACTIONS, checkpoint_data, SIZE, explicit_action_mask)
elif mode == 'ppo_conv_ext_deep':
    solver = PPOConvExtDeepSolver(info, USED_ACTIONS, checkpoint_data, SIZE, explicit_action_mask)
else:
    raise ValueError(f'unknown mode: "{mode}"')

In [52]:
np_output = solver._calculate_output([e.to_dict() for e in env._get_entites(0)], player_pos)

In [53]:
np_output

array([0.01116721, 0.31733118, 0.00610177, 0.23641222, 0.13175212,
       0.10487788, 0.05623798, 0.13611964])

In [54]:
model_output = agent.model(tensor_data)['policy'].detach().cpu().numpy()

In [55]:
model_output

array([[0.01116721, 0.31733114, 0.00610177, 0.23641221, 0.13175212,
        0.10487789, 0.05623798, 0.13611963]], dtype=float32)

In [56]:
# data2, data1 = zip(*weights.items())

# data1 = pkl.dumps(data1)
# data2 = pkl.dumps(data2)

In [57]:
data1 = []
data2 = []
for k, v in weights.items():
    if 'num_batches_tracked' in k:
        continue
    print(k, v.shape)
    data2.append(k)
    buffer = io.BytesIO()
    v = v.astype(np.float16)
    np.save(buffer, v)
    data1.append(buffer.getvalue())
    # data1.append(zlib.compress(buffer.getvalue(), level=9))

data1 = pkl.dumps(data1)
data2 = pkl.dumps(data2)

encoder.mconv_list.0.ones_kernel (1, 1, 3, 3)
encoder.mconv_list.0.conv.weight (8, 21, 3, 3)
encoder.mconv_list.0.conv.bias (8,)
encoder.mconv_list.1.ones_kernel (1, 1, 5, 5)
encoder.mconv_list.1.conv.weight (8, 21, 5, 5)
encoder.mconv_list.1.conv.bias (8,)
encoder.ln_list.0.weight (8,)
encoder.ln_list.0.bias (8,)
encoder.ln_list.1.weight (8,)
encoder.ln_list.1.bias (8,)
encoder.fc_list.0.weight (16, 8)
encoder.fc_list.0.bias (16,)
encoder.fc_list.1.weight (16, 8)
encoder.fc_list.1.bias (16,)
actor.weight (8, 16)
actor.bias (8,)
critic.weight (1, 16)
critic.bias (1,)
terminator.weight (1, 16)
terminator.bias (1,)
occupation_head.weight (1, 16)
occupation_head.bias (1,)


In [58]:
with open('../src/game/template.py') as f:
    lines = f.readlines()

In [59]:
with open('../src/game/template_submit.py', 'w') as f:
    for line in lines:
        line = line.replace("b'data1data1data1'", str(base64.b64encode(zlib.compress(data1, level=9))))
        line = line.replace("b'data2data2data2'", str(base64.b64encode(zlib.compress(data2, level=9))))
        line = line.replace("mode = 'mode'", f"mode = '{mode}'")
        line = line.replace("USED_ACTIONS = DEFAULT_KUTULU_ACTIONS", "USED_ACTIONS = EXTENDED_KUTULU_ACTIONS")
        line = line.replace("SIZE = 3", f"SIZE = {agent.size}")
        line = line.replace(
            "ACTION_MASK = np.ones_like(USED_ACTIONS)",
            f"ACTION_MASK = np.array({agent.explicit_action_mask.tolist()})",
        )
        line = line.replace("experiment: ''", f"experiment: '{experiment}'")
        f.write(line)

In [60]:
!ls -lh ../src/game/template_submit.py

-rw-r--r--  1 aleksei  staff    76K 30 ноя 10:29 ../src/game/template_submit.py
